<a href="https://colab.research.google.com/github/salphonseds/llm-from-scratch/blob/main/notebooks/06_Generation_Strategies.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Setup + recreate the GPT model from 04 notebook - Same as 05 notebook
import torch
import torch.nn as nn
import math

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.shape
        Q = self.W_q(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)

        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = torch.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        out = attn_weights @ V
        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        return self.W_o(out)


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_ff, d_model), nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        x = x + self.dropout(self.attn(self.ln1(x), mask))
        x = x + self.dropout(self.ff(self.ln2(x)))
        return x


class GPTModel(nn.Module):
    def __init__(self, vocab_size=50257, d_model=512, num_heads=8, d_ff=2048,
                 num_layers=6, max_len=256, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.dropout = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)
        ])

        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight  # weight tying

        self.register_buffer("causal_mask",
            torch.tril(torch.ones(max_len, max_len)).view(1, 1, max_len, max_len))

    def forward(self, idx):
        batch_size, seq_len = idx.shape
        pos = torch.arange(seq_len, device=idx.device).unsqueeze(0)
        x = self.token_emb(idx) + self.pos_emb(pos)
        x = self.dropout(x)
        mask = self.causal_mask[:, :, :seq_len, :seq_len]
        for block in self.blocks:
            x = block(x, mask)
        x = self.ln_f(x)
        return self.lm_head(x)


# Recreate model with your established hyperparameters
torch.manual_seed(42)
model = GPTModel(vocab_size=50257, d_model=512, num_heads=8, d_ff=2048,
                  num_layers=6, max_len=256).to(device)

num_params = sum(p.numel() for p in model.parameters())
print(f"Model recreated: {num_params:,} parameters")
print(f"Expected: 44,777,984 parameters")
print(f"Match: {num_params == 44_777_984}")

Using device: cuda
Model recreated: 44,777,984 parameters
Expected: 44,777,984 parameters
Match: True


In [ ]:
print('model' in dir())

True


In [ ]:
# Cell 2: Init fix + tokenizer + corpus + quick retrain (recovering Notebook 05's trained state)
import tiktoken

def _init_weights(module):
    if isinstance(module, nn.Linear):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if module.bias is not None:
            nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)

model.apply(_init_weights)

tokenizer = tiktoken.get_encoding("gpt2")
text = """
The old lighthouse stood at the edge of the cliff, its light sweeping
across the dark water every few seconds. Sailors who passed that coast
said the keeper never slept, that he watched the horizon for ships that
would never come. Every winter the storms grew fiercer, and every winter
the lighthouse held its ground. The keeper would climb the spiral stairs
each evening, counting them as he always had, and light the lamp before
the last color drained from the sky. He believed that somewhere out on
the water, someone was always looking back toward the light, grateful
for the proof that land still existed, that the world had not been
swallowed entirely by the sea.
"""
tokens = tokenizer.encode(text)
print(f"Token count: {len(tokens)}")

seq_len, stride = 10, 5
inputs, targets = [], []
for i in range(0, len(tokens) - seq_len, stride):
    inputs.append(tokens[i:i+seq_len])
    targets.append(tokens[i+1:i+seq_len+1])
inputs = torch.tensor(inputs).to(device)
targets = torch.tensor(targets).to(device)
print(f"Training examples: {len(inputs)}")

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
loss_fn = nn.CrossEntropyLoss()
model.train()

for epoch in range(100):
    perm = torch.randperm(len(inputs))
    for i in range(0, len(inputs), 4):
        idx = perm[i:i+4]
        logits = model(inputs[idx])
        loss = loss_fn(logits.view(-1, 50257), targets[idx].view(-1))
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
    if epoch % 20 == 0 or epoch == 99:
        print(f"Epoch {epoch:3d} | Loss: {loss.item():.4f}")

model.eval()
print(f"\nRetrained and ready. Final loss: {loss.item():.4f}")

Token count: 146
Training examples: 28
Epoch   0 | Loss: 9.3401
Epoch  20 | Loss: 0.1434
Epoch  40 | Loss: 0.0108
Epoch  60 | Loss: 0.0533
Epoch  80 | Loss: 0.0662
Epoch  99 | Loss: 0.0752

Retrained and ready. Final loss: 0.0752


In [ ]:
# Cell 3: Greedy decoding (baseline)
def generate_greedy(model, prompt_ids, max_new_tokens=20):
    model.eval()
    idx = torch.tensor([prompt_ids]).to(device)
    with torch.no_grad():
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -256:]  # respect max_len context window
            logits = model(idx_cond)
            next_logits = logits[:, -1, :]  # logits for the last position
            next_token = torch.argmax(next_logits, dim=-1, keepdim=True)
            idx = torch.cat([idx, next_token], dim=1)
    return idx[0].tolist()

# Prompt: start of the corpus, so the model is on familiar ground
prompt_text = "The old lighthouse stood"
prompt_ids = tokenizer.encode(prompt_text)

output_ids = generate_greedy(model, prompt_ids, max_new_tokens=20)
print("Prompt:", repr(prompt_text))
print("Generated:", repr(tokenizer.decode(output_ids)))

Prompt: 'The old lighthouse stood'
Generated: 'The old lighthouse stood the held its its light sweeping\nthe sweeping\nthe dark dark water water every water water every few'


In [ ]:
# Cell 4: Temperature sampling
def generate_temperature(model, prompt_ids, max_new_tokens=20, temperature=1.0):
    model.eval()
    idx = torch.tensor([prompt_ids]).to(device)
    with torch.no_grad():
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -256:]
            logits = model(idx_cond)
            next_logits = logits[:, -1, :] / temperature
            probs = torch.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_token], dim=1)
    return idx[0].tolist()

prompt_text = "The old lighthouse stood"
prompt_ids = tokenizer.encode(prompt_text)

torch.manual_seed(42)
for temp in [0.5, 1.0, 1.5]:
    output_ids = generate_temperature(model, prompt_ids, max_new_tokens=20, temperature=temp)
    print(f"T={temp}: {tokenizer.decode(output_ids)!r}")
    print()

T=0.5: 'The old lighthouse stood the held its its light sweeping\n sweeping\nthe dark sweeping\nthe dark dark water water everythe'

T=1.0: 'The old lighthouse stoodallowed entirely by the sea.\n the sea. Every winter the sea. slept The Every winter the'

T=1.5: 'The old lighthouse stood disse * Runument Endurance Gators at the\nthe ankles coachinginventory had the KramerJanepres Wrestling 317'



In [ ]:
# Cell 5: Top-k sampling
def generate_topk(model, prompt_ids, max_new_tokens=20, temperature=1.0, k=10):
    model.eval()
    idx = torch.tensor([prompt_ids]).to(device)
    with torch.no_grad():
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -256:]
            logits = model(idx_cond)
            next_logits = logits[:, -1, :] / temperature

            # Keep only the top-k logits, mask everything else to -inf
            topk_vals, topk_idx = torch.topk(next_logits, k)
            mask = torch.full_like(next_logits, float('-inf'))
            mask.scatter_(1, topk_idx, topk_vals)

            probs = torch.softmax(mask, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_token], dim=1)
    return idx[0].tolist()

prompt_text = "The old lighthouse stood"
prompt_ids = tokenizer.encode(prompt_text)

torch.manual_seed(42)
for k in [5, 10, 20]:
    output_ids = generate_topk(model, prompt_ids, max_new_tokens=20, temperature=1.0, k=k)
    print(f"k={k}: {tokenizer.decode(output_ids)!r}")
    print()

k=5: 'The old lighthouse stoodallowed entirely by the sea.\n the sea. sea. sea. Every winter the sea The sea'

k=10: 'The old lighthouse stoodallowed entirely by the sea.\n the sea. Every winter the sea. sea The Every winter the'

k=20: 'The old lighthouse stood the held its its light sweeping\nthe dark sweeping\nthe dark water water every few dark water every'



In [ ]:
# Cell 5b: Top-k sampling, properly controlled (reset seed before each run)
prompt_text = "The old lighthouse stood"
prompt_ids = tokenizer.encode(prompt_text)

for k in [5, 10, 20]:
    torch.manual_seed(42)  # reset before EACH call, not once before the loop
    output_ids = generate_topk(model, prompt_ids, max_new_tokens=20, temperature=1.0, k=k)
    print(f"k={k}: {tokenizer.decode(output_ids)!r}")
    print()

k=5: 'The old lighthouse stoodallowed entirely by the sea.\n the sea. sea. sea. Every winter the sea The sea'

k=10: 'The old lighthouse stoodallowed entirely by the sea.\n the sea.ier evening, and light Every winter the sea sea'

k=20: 'The old lighthouse stoodallowed entirely by the sea.\n the sea.ier evening, and light edge sea. Every winter'



In [ ]:
# Cell 6: Top-p (nucleus) sampling
def generate_topp(model, prompt_ids, max_new_tokens=20, temperature=1.0, p=0.9):
    model.eval()
    idx = torch.tensor([prompt_ids]).to(device)
    with torch.no_grad():
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -256:]
            logits = model(idx_cond)
            next_logits = logits[:, -1, :] / temperature
            probs = torch.softmax(next_logits, dim=-1)

            # Sort probabilities descending, find cumulative sum
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

            # Keep tokens up to and including the one that crosses p
            cutoff = cumulative_probs > p
            cutoff[..., 1:] = cutoff[..., :-1].clone()  # shift right: always keep at least 1 token
            cutoff[..., 0] = False
            sorted_probs[cutoff] = 0.0

            # Renormalize and map back to original token ids
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_in_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_token = sorted_idx.gather(-1, next_in_sorted)

            idx = torch.cat([idx, next_token], dim=1)
    return idx[0].tolist()

prompt_text = "The old lighthouse stood"
prompt_ids = tokenizer.encode(prompt_text)

for p in [0.5, 0.9, 0.99]:
    torch.manual_seed(42)
    output_ids = generate_topp(model, prompt_ids, max_new_tokens=20, temperature=1.0, p=p)
    print(f"p={p}: {tokenizer.decode(output_ids)!r}")
    print()

p=0.5: 'The old lighthouse stood the held its its light sweeping\nthe sweeping\nthe dark dark water water every water water every few'

p=0.9: 'The old lighthouse stood the held its its ground. The The The The keeper would climb spiral spiral stairs spiral stairs\nthe'

p=0.99: 'The old lighthouse stood the held its its ground. The The The The keeper would climb spiral spiral stairs spiral stairs\nthe'



In [ ]:
# Cell 6b: Inspect the actual probability distribution at the first generation step
idx = torch.tensor([prompt_ids]).to(device)
with torch.no_grad():
    logits = model(idx)
    probs = torch.softmax(logits[:, -1, :], dim=-1)
    sorted_probs, sorted_idx = torch.sort(probs, descending=True)

print("Top 10 token probabilities at this step:")
for i in range(10):
    tok = tokenizer.decode([sorted_idx[0, i].item()])
    print(f"  {i+1}. {tok!r:15s} prob={sorted_probs[0, i].item():.6f}")

print(f"\nCumulative prob of top 1: {sorted_probs[0,0].item():.4f}")
print(f"Cumulative prob of top 3: {sorted_probs[0,:3].sum().item():.4f}")
print(f"Cumulative prob of top 5: {sorted_probs[0,:5].sum().item():.4f}")

Top 10 token probabilities at this step:
  1. ' the'          prob=0.353266
  2. ' at'           prob=0.206565
  3. 'allowed'       prob=0.162335
  4. ' always'       prob=0.044019
  5. ' existed'      prob=0.040295
  6. ' stood'        prob=0.022793
  7. '.'             prob=0.016130
  8. ' cliff'        prob=0.011893
  9. ' ships'        prob=0.010543
  10. ' out'          prob=0.007402

Cumulative prob of top 1: 0.3533
Cumulative prob of top 3: 0.7222
Cumulative prob of top 5: 0.8065


In [ ]:
# Cell 6c: Inspect the distribution a few tokens into generation, not just step 1
context_text = "The old lighthouse stood the held its its light sweeping"
context_ids = tokenizer.encode(context_text)
idx = torch.tensor([context_ids]).to(device)

with torch.no_grad():
    logits = model(idx)
    probs = torch.softmax(logits[:, -1, :], dim=-1)
    sorted_probs, sorted_idx = torch.sort(probs, descending=True)

print("Top 5 token probabilities at this later step:")
for i in range(5):
    tok = tokenizer.decode([sorted_idx[0, i].item()])
    print(f"  {i+1}. {tok!r:15s} prob={sorted_probs[0, i].item():.6f}")

Top 5 token probabilities at this later step:
  1. '\n'            prob=0.999444
  2. ' on'           prob=0.000023
  3. ' world'        prob=0.000014
  4. 'ier'           prob=0.000012
  5. ' out'          prob=0.000012


In [ ]:
# Cell 7: Beam search
def generate_beam(model, prompt_ids, max_new_tokens=20, beam_width=3):
    model.eval()
    # Each beam: (token_id_list, cumulative_log_prob)
    beams = [(prompt_ids, 0.0)]

    with torch.no_grad():
        for _ in range(max_new_tokens):
            candidates = []
            for seq, score in beams:
                idx = torch.tensor([seq[-256:]]).to(device)
                logits = model(idx)
                log_probs = torch.log_softmax(logits[0, -1, :], dim=-1)

                # Only consider top beam_width next tokens per beam (keeps expansion manageable)
                topk_log_probs, topk_ids = torch.topk(log_probs, beam_width)
                for lp, tid in zip(topk_log_probs.tolist(), topk_ids.tolist()):
                    candidates.append((seq + [tid], score + lp))

            # Keep only the best beam_width sequences overall, across all expansions
            candidates.sort(key=lambda x: x[1], reverse=True)
            beams = candidates[:beam_width]

    # Return the highest-scoring beam
    best_seq, best_score = beams[0]
    return best_seq, best_score

prompt_text = "The old lighthouse stood"
prompt_ids = tokenizer.encode(prompt_text)

for bw in [1, 3, 5]:
    output_ids, score = generate_beam(model, prompt_ids, max_new_tokens=20, beam_width=bw)
    print(f"beam_width={bw} (score={score:.2f}): {tokenizer.decode(output_ids)!r}")
    print()

beam_width=1 (score=-6.77): 'The old lighthouse stood the held its its light sweeping\nthe sweeping\nthe dark dark water water every water water every few'

beam_width=3 (score=-6.77): 'The old lighthouse stood the held its its light sweeping\nthe sweeping\nthe dark dark water water every water water every few'

beam_width=5 (score=-6.74): 'The old lighthouse stoodallowed entirely by the sea.\n the sea. sea. Every winter winter the sea. Every winter'



In [ ]:
# Cell 8: Compare all strategies on a prompt NOT in the training corpus
novel_prompt = "On a clear summer morning"
novel_ids = tokenizer.encode(novel_prompt)

print(f"Prompt: {novel_prompt!r}\n")

torch.manual_seed(42)
greedy_out = generate_greedy(model, novel_ids, max_new_tokens=20)
print(f"Greedy:      {tokenizer.decode(greedy_out)!r}\n")

torch.manual_seed(42)
temp_out = generate_temperature(model, novel_ids, max_new_tokens=20, temperature=1.0)
print(f"Temp (T=1):  {tokenizer.decode(temp_out)!r}\n")

torch.manual_seed(42)
topk_out = generate_topk(model, novel_ids, max_new_tokens=20, temperature=1.0, k=10)
print(f"Top-k (k=10):{tokenizer.decode(topk_out)!r}\n")

torch.manual_seed(42)
topp_out = generate_topp(model, novel_ids, max_new_tokens=20, temperature=1.0, p=0.9)
print(f"Top-p (p=0.9):{tokenizer.decode(topp_out)!r}\n")

beam_out, beam_score = generate_beam(model, novel_ids, max_new_tokens=20, beam_width=3)
print(f"Beam (w=3):  {tokenizer.decode(beam_out)!r} (score={beam_score:.2f})")

Prompt: 'On a clear summer morning'

Greedy:      'On a clear summer morning always had, and light the keeper and light the light the light the lamp before\n every every every'

Temp (T=1):  'On a clear summer morning seconds. He believed that somewhere somewhere somewhere out had He believed that somewhere outier that somewhere out on'

Top-k (k=10):'On a clear summer morning always had, and light the keeper and light the and light the lamp before\n every every every every'

Top-p (p=0.9):'On a clear summer morning always had, and light the light the keeper and light the keeper light the lamp light winter\nthe'

Beam (w=3):  'On a clear summer morningsaid keeper would climb the keeper never would climb the keeper would climb climb the keeper would climb climb climb' (score=-6.34)


In [ ]:
# Cell 9: Day 10 Summary
print("=" * 60)
print("Day 10 Complete: Generation Strategies")
print("=" * 60)

print("\n✓ Implemented and compared 5 decoding strategies:")
print("  1. Greedy decoding (argmax) — deterministic, prone to loops")
print("  2. Temperature sampling — controls distribution sharpness")
print("  3. Top-k sampling — fixed candidate pool, excludes noisy tail")
print("  4. Top-p (nucleus) sampling — adaptive pool size")
print("  5. Beam search — tracks multiple sequences, optimizes")
print("     cumulative score (not necessarily least repetitive)")

print("\n🔑 Key concepts mastered:")
print("  • Decoding strategy reshapes output, but can't add capability")
print("    the model wasn't trained to have")
print("  • High temperature on an undertrained model surfaces noise")
print("    from never-updated logits, not creative diversity")
print("  • Top-p degenerates toward greedy when the model's output")
print("    distribution is already near one-hot (common in overfitting)")
print("  • Wide beam search can find higher-scoring paths that are")
print("    still repetitive — optimal score != non-repetitive text")
print("  • An overfit model regurgitates training data even on")
print("    out-of-distribution prompts, rather than degrading gracefully")

print("\n📊 Notable result:")
print(f"  Beam search (w=5) found a higher-scoring sequence than")
print(f"  greedy/beam(w=1) or beam(w=3): -6.74 vs -6.77")

print("\n🎯 Next up: Days 11-12 - Modern Architectures")
print("  - RMSNorm, SwiGLU, RoPE (LLaMA-style components)")
print("  - Grouped-query attention (GQA)")
print("  - Flash Attention concept")

print("\n" + "=" * 60)
print("10/21 days complete (~48%) - save this notebook!")
print("=" * 60)

Day 10 Complete: Generation Strategies

✓ Implemented and compared 5 decoding strategies:
  1. Greedy decoding (argmax) — deterministic, prone to loops
  2. Temperature sampling — controls distribution sharpness
  3. Top-k sampling — fixed candidate pool, excludes noisy tail
  4. Top-p (nucleus) sampling — adaptive pool size
  5. Beam search — tracks multiple sequences, optimizes
     cumulative score (not necessarily least repetitive)

🔑 Key concepts mastered:
  • Decoding strategy reshapes output, but can't add capability
    the model wasn't trained to have
  • High temperature on an undertrained model surfaces noise
    from never-updated logits, not creative diversity
  • Top-p degenerates toward greedy when the model's output
    distribution is already near one-hot (common in overfitting)
  • Wide beam search can find higher-scoring paths that are
    still repetitive — optimal score != non-repetitive text
  • An overfit model regurgitates training data even on
    out-of-distri